# ⚠️ Setup Instructions

To run this notebook successfully, please ensure you complete the following steps:

1. **Switch to T4 GPU**: Ensure your runtime is configured to use a T4 GPU. In the menu, go to **Runtime > Change runtime type**, select **T4 GPU** under *Hardware accelerator*, and click **Save**.
2. **Shared Folder Shortcut**: Make sure you have access to the shared repository Drive folder and have created a **shortcut** from it directly to your own Google Drive's home directory (`My Drive`). This allows the code to resolve the path `/content/drive/MyDrive/traffic-risk-agent-shared/scene-risk-traffic-safety-analysis` correctly.
3. **Drive Permissions**: When you select **Runtime > Run all**, Google Colab will prompt you for Google Drive access permissions. You must authorize full access to your Drive for the file reading and writing processes to run without interruption.

# Traffic Risk Agent Pipeline
This notebook walks you through setting up and running the Traffic Risk Agent Pipeline using YOLO spatial metadata extraction and an Ollama-hosted vision-language model (Moondream) to assess traffic scene risk.

## Cell 1: Team Instructions & Drive Mount
We will mount Google Drive, check GPU status, and configure the working directory to access our shared repository.

In [1]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Check GPU availability
!nvidia-smi

# Set working directory to project repository
REPO_PATH = '/content/drive/MyDrive/traffic-risk-agent-shared/scene-risk-traffic-safety-analysis'
os.chdir(REPO_PATH)
print(f"Current Working Directory: {os.getcwd()}")

# List directory contents
print("Directory contents:")
print(os.listdir('.'))

Mounted at /content/drive
Wed Oct 14 19:02:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------

## Cell 2: Install & Start Ollama Background Daemon
We will install Ollama, launch it as a background service, wait for initialization, and pull the `moondream` vision model.

In [2]:
import os
import subprocess
import time
import urllib.request

# 1. Kill any stale Ollama daemon running in background memory
print("Cleaning up any existing Ollama processes...")
!pkill -9 -f ollama
time.sleep(2)

# 2. Install/Update hardware tools and the latest Ollama release
!sudo apt-get update && sudo apt-get install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Check installed Ollama version (must be >= 0.4.0)
!/usr/local/bin/ollama --version

# 4. Set CUDA environment variables
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:/usr/local/cuda/lib64:" + env.get("LD_LIBRARY_PATH", "")

# 5. Start a FRESH Ollama daemon in the background
print("Starting fresh Ollama daemon with GPU support...")
process = subprocess.Popen(
    ["/usr/local/bin/ollama", "serve"],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# 6. Wait for the new server to bind to port 11434
print("Waiting for Ollama server to become available...")
for i in range(15):
    try:
        urllib.request.urlopen("http://localhost:11434")
        print("Ollama server is active and running successfully with GPU support!")
        break
    except Exception:
        time.sleep(2)
else:
    print("Ollama daemon failed to start.")

# 7. Pull llama3.2-vision
print("Pulling llava:7b...")
!/usr/local/bin/ollama pull llava:7b

# 8. List installed models
!/usr/local/bin/ollama list

Cleaning up any existing Ollama processes...
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:4 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Get:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [1,919 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:9 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [1,409 kB]
Get:10 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:12 http://security.ubuntu.com/ubuntu noble-security/restricted amd64 Packages 

## Cell 3: Install Python Dependencies
We will install the required Python libraries using standard Colab pip commands.

In [3]:
# Fallback for uv package manager:
# !pip install uv && !uv sync

!pip install ultralytics opencv-python pandas scipy scikit-learn ollama

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.9/96.9 kB 10.6 MB/s eta 0:00:00


## Cell 4: Single-Image Sanity Check Test
We will run a dry-run test on a sample image to verify YOLO extraction and Ollama VLM responses.

In [4]:
# import cv2
# import glob
# from ultralytics import YOLO
# import ollama

# # Find a sample image
# image_files = glob.glob('data/raw_images/*')
# if not image_files:
#     print("No images found in data/raw_images/. Please check paths.")
# else:
#     sample_img_path = image_files[0]
#     print(f"Testing with image: {sample_img_path}")

#     # 1. Run YOLO Object Detection
#     yolo_model = YOLO('yolo26n.pt')
#     results = yolo_model(sample_img_path)
#     boxes = results[0].boxes
#     detected_objects = [yolo_model.names[int(cls)] for cls in boxes.cls]
#     print(f"Detected objects: {detected_objects}")

#     # 2. Query Ollama llama3.2-vision Model
#     prompt = "Analyze the traffic risk of this scene. Provide a JSON output with keys: 'Confidence' (number 0-100) and 'Meaning' (risk description)."
#     try:
#         response = ollama.generate(
#             model='llava:7b',
#             prompt=prompt,
#             images=[sample_img_path]
#         )
#         print("\nOllama Response:")
#         print(response['response'])
#     except Exception as e:
#         print(f"Error connecting to Ollama API: {e}")

## Cell 5: Full Pipeline Execution
Now we will run the main evaluation pipeline. Results are written directly to `output/experiment_results.csv` on your persistent Drive storage.

In [5]:
# Execute the pipeline using the much stronger llama3.2-vision model
!python main.py --provider ollama --model llava:7b

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Found 58 image(s) in /content/drive/MyDrive/traffic-risk-agent-shared/scene-risk-traffic-safety-analysis/data/raw_images

Loaded 0 ground-truth entries.

Initialising YOLO perception extractor …
Initialising VLM client (ollama) …

[1/58] Processing: 0.png
  YOLO: 4 road user(s) detected
  VLM Condition A (image-only) …
    → Score: 0  |  This is a static scene with no moving vehicles or pedestrians. No driver attenti
  VLM Condition B (image + YOLO) …
    → Score: 0  |  Moderate attention required due to the presence of pedestrians and the need to b

[2/58] Processing: 1.png
  YOLO: 6 road user(s) detected
  VLM Condition A (image-only) …
    → Score: 0  |  Clear road with no obstacles or hazards r

In [6]:
# # Let's inspect main.py to understand how it processes, parses, and handles errors/scores from the VLM
# with open('main.py', 'r') as f:
#     lines = f.readlines()

# # Search for VLM querying, prompt logic, or error handling blocks
# for i, line in enumerate(lines):
#     if 'class' in line or 'def ' in line or 'Score' in line or 'ollama' in line.lower() or '-1' in line:
#         print(f"Line {i+1}: {line.strip()}")

## Cell 6: Results Visualization & Evaluation Metrics
Finally, we will load the pipeline run results, compare predictions with ground truth, and compute evaluation metrics (Pearson, Spearman, and Classification scores).

In [7]:
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import precision_score, recall_score, f1_score

results_path = 'output/experiment_results.csv'
try:
    df = pd.read_csv(results_path)
    print("Preview of experiment results:")
    display(df.head())

    if 'ground_truth' in df.columns and 'predicted_risk' in df.columns:
        # Continuous Evaluation Metrics
        r_coef, _ = pearsonr(df['predicted_risk'], df['ground_truth'])
        rho_coef, _ = spearmanr(df['predicted_risk'], df['ground_truth'])
        print(f"\nPearson Correlation (r): {r_coef:.3f}")
        print(f"Spearman Rank Correlation (rho): {rho_coef:.3f}")

        # Classification Evaluation Metrics (Threshold >= 70)
        threshold = 70
        y_true = (df['ground_truth'] >= threshold).astype(int)
        y_pred = (df['predicted_risk'] >= threshold).astype(int)

        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)

        print(f"\nClassification metrics (Threshold >= {threshold}):")
        print(f"Precision: {precision:.3f}")
        print(f"Recall:    {recall:.3f}")
        print(f"F1 Score:  {f1:.3f}")
    else:
        print("Columns 'ground_truth' or 'predicted_risk' missing from the results csv.")
except FileNotFoundError:
    print(f"Results file not found at {results_path}. Ensure the pipeline completed successfully.")

Preview of experiment results:


,image_id,yolo_detections,yolo_summary,condition_a_score,condition_a_meaning,condition_b_score,condition_b_meaning,ground_truth_score
0,0,4,Detected Road Users: 4 pedestrians.\n - 1 ped...,0,This is a static scene with no moving vehicles...,0,Moderate attention required due to the presenc...,NaN
1,1,6,"Detected Road Users: 4 cars, 2 pedestrians.\n ...",0,Clear road with no obstacles or hazards requir...,70,The driver needs to be vigilant due to the pre...,NaN
2,10,7,Detected Road Users: 7 cars.\n - 1 car (large...,0,"Calm/clear road, no driver attention required.",80,The driver needs to be vigilant due to the pre...,NaN
3,100,0,Detected Road Users: None.,0,"The image is a still from a video, showing a c...",0,The image depicts an empty road with no other ...,NaN
4,102,4,Detected Road Users: 4 cars.\n - 1 car (small...,0,No driver attention required in this scene. Th...,0,The presence of multiple cars in the scene sug...,NaN


Columns 'ground_truth' or 'predicted_risk' missing from the results csv.
